In [5]:
!pip install torch torchvision torchaudio pandas matplotlib tqdm
!pip install torchsummary kaggle


  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 23.8 MB/s eta 0:00:00
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\pra72\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


  Using cached python_slugify-8.0.4-py2.py3-none-any.whl.metadata (8.5 kB)
  Using cached text_unidecode-1.3-py2.py3-none-any.whl.metadata (2.4 kB)
Using cached python_slugify-8.0.4-py2.py3-none-any.whl (10 kB)
Using cached text_unidecode-1.3-py2.py3-none-any.whl (78 kB)



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\pra72\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
# --- SETUP AND IMPORTS ---
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import zipfile # Added for zipfile operations
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.utils import make_grid
from torchvision.datasets import ImageFolder
from torchsummary import summary
from tqdm.notebook import tqdm # Make sure this is imported if running in a notebook


# --- KAGGLE SETUP ---
def setup_kaggle():
    """Setup Kaggle API and download dataset"""
    # Install required packages
    print("Installing required packages...")
    # In a notebook, use !pip. In a .py script, these should be installed beforehand.
    # For this example, I'll assume they are installed or handle it externally.
    # If running in Colab:
    try:
        import google.colab
        IN_COLAB = True
    except:
        IN_COLAB = False

    if IN_COLAB:
        !pip install -q kaggle torchsummary tqdm
        from google.colab import files
        # Upload your kaggle.json file
        print("\nPlease upload your kaggle.json file...")
        uploaded = files.upload()
        if not uploaded:
            print("Kaggle.json not uploaded. Exiting setup.")
            return None, None, None

        # Setup Kaggle credentials
        print("\nSetting up Kaggle credentials...")
        os.makedirs('/root/.kaggle', exist_ok=True)
        !cp kaggle.json /root/.kaggle/
        !chmod 600 /root/.kaggle/kaggle.json
    else:
        print("Not in Colab. Assuming Kaggle API is configured and packages are installed.")
        if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
            print("Kaggle API credentials not found. Please set them up manually.")
            print("Skipping download. Ensure dataset is in './plant_diseases'")
            # return None, None, None # Or set default paths if dataset is expected locally

    # Download the dataset
    dataset_zip_file = "new-plant-diseases-dataset.zip"
    dataset_extract_path = "plant_diseases"

    if not os.path.exists(dataset_zip_file) and IN_COLAB: # Only download if not present and in Colab
        print("\nDownloading the dataset from Kaggle (this may take a few minutes)...")
        !kaggle datasets download -d vipoooool/new-plant-diseases-dataset
    elif not os.path.exists(dataset_zip_file) and not IN_COLAB:
        print(f"{dataset_zip_file} not found. Please download it manually or ensure Kaggle API is set up.")
        # return None, None, None # Or proceed assuming it's already extracted

    # Extract the dataset with progress indicator
    if os.path.exists(dataset_zip_file) and not os.path.exists(os.path.join(dataset_extract_path, "New Plant Diseases Dataset(Augmented)")):
        print("\nExtracting the dataset...")
        # Get the size of the zip file
        zip_size = os.path.getsize(dataset_zip_file)

        with zipfile.ZipFile(dataset_zip_file, 'r') as zip_ref:
            # Create extraction progress bar
            members = zip_ref.infolist()
            extract_bar = tqdm(total=len(members), desc="Extracting files")

            # Extract each file with progress update
            for member in members:
                zip_ref.extract(member, dataset_extract_path)
                extract_bar.update(1)
        extract_bar.close()
    elif os.path.exists(os.path.join(dataset_extract_path, "New Plant Diseases Dataset(Augmented)")):
        print("Dataset already extracted.")
    else:
        print("Dataset zip file not found, and dataset not extracted. Please check paths.")
        # return None, None, None

    # Set data directory paths
    data_dir = os.path.join(dataset_extract_path, "New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)")
    train_dir = os.path.join(data_dir, "train")
    valid_dir = os.path.join(data_dir, "valid")

    if not os.path.exists(train_dir):
        print(f"Train directory not found: {train_dir}")
        print("Please check the dataset structure after extraction.")
        return None, None, None

    print(f"\nDataset setup complete. Data directory: {data_dir}")
    return data_dir, train_dir, valid_dir

# --- DATA EXPLORATION ---
def explore_data(train_dir):
    """Explore the dataset and return basic statistics"""
    diseases = sorted(os.listdir(train_dir)) # Sort for consistent class ordering

    # Count unique plants
    plants = sorted(list(set([plant.split('___')[0] for plant in diseases])))

    # Count images per class
    nums = {}
    for disease in diseases:
        nums[disease] = len(os.listdir(os.path.join(train_dir, disease)))

    # Create a dataframe for image distribution
    img_per_class = pd.DataFrame(nums.values(), index=nums.keys(), columns=["no. of images"])

    # Print summary
    print(f"Number of classes: {len(diseases)}")
    print(f"Unique plants: {', '.join(plants)}")
    print(f"Total training images: {sum(nums.values()):,}")
    print("\nImage distribution per class (top 5):")
    print(img_per_class.sort_values(by="no. of images", ascending=False).head())


    return diseases, nums, plants, img_per_class

# --- DATA VISUALIZATION ---
def visualize_data(nums, diseases, train_dataset_for_samples): # Renamed for clarity
    """Visualize dataset information"""
    # Plot class distribution
    plt.figure(figsize=(20, 7)) # Adjusted figsize
    plt.bar(range(len(nums)), list(nums.values()), width=0.8, color='skyblue')
    plt.xlabel('Plants/Diseases', fontsize=12)
    plt.ylabel('Number of images', fontsize=12)
    plt.xticks(range(len(nums)), diseases, fontsize=8, rotation=90) # Increased font size
    plt.title('Images per each class of plant disease', fontsize=15)
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.show()

    # Show a sample image
    def show_image(image, label_idx, classes_list):
        print(f"Label: {classes_list[label_idx]} (Class {label_idx})")
        plt.figure(figsize=(5, 5)) # Slightly larger
        # Inverse normalize for display if normalization was applied
        # Assuming standard ImageNet normalization
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image_display = image * std + mean
        image_display = torch.clamp(image_display, 0, 1) # Clamp values to [0,1]
        plt.imshow(image_display.permute(1, 2, 0))
        plt.axis('off')
        plt.show()

    # Display a random sample
    if len(train_dataset_for_samples) > 0:
        sample_idx = np.random.randint(0, len(train_dataset_for_samples))
        img, label_idx = train_dataset_for_samples[sample_idx]
        show_image(img, label_idx, train_dataset_for_samples.classes)
    else:
        print("Training dataset is empty, cannot show sample image.")


    # Show a batch of images
    def show_batch(dataloader):
        for images, _ in dataloader: # We don't need labels here
            fig, ax = plt.subplots(figsize=(14, 10)) # Adjusted figsize
            ax.set_xticks([])
            ax.set_yticks([])
            # Inverse normalize for display
            mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
            images_display = images * std + mean
            images_display = torch.clamp(images_display, 0, 1)
            ax.imshow(make_grid(images_display[:16], nrow=4).permute(1, 2, 0))
            plt.show()
            break
    return show_batch


# --- DEVICE CONFIGURATION ---
def get_device():
    """Get the device (GPU if available, otherwise CPU)"""
    if torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")

def to_device(data, device):
    """Move tensor(s) to the selected device"""
    if isinstance(data, (list, tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)

class DeviceDataLoader():
    """Wrap a dataloader to move data to a device"""
    def __init__(self, dl, device):
        self.dl = dl
        self.device = device

    def __iter__(self):
        """Yield a batch of data after moving it to device"""
        for batch in self.dl:
            yield to_device(batch, self.device)

    def __len__(self):
        """Return the number of batches"""
        return len(self.dl)

# --- MODEL ARCHITECTURE ---
class ImageClassificationBase(nn.Module):
    def training_step(self, batch):
        images, labels = batch
        out = self(images)
        loss = F.cross_entropy(out, labels)
        return loss

    def validation_step(self, batch):
        images, labels = batch
        out = self(images)
        loss = F.cross_entropy(out, labels)
        acc = accuracy(out, labels)
        return {"val_loss": loss.detach(), "val_accuracy": acc}

    def validation_epoch_end(self, outputs):
        batch_losses = [x["val_loss"] for x in outputs]
        batch_accuracy = [x["val_accuracy"] for x in outputs]
        epoch_loss = torch.stack(batch_losses).mean()
        epoch_accuracy = torch.stack(batch_accuracy).mean()
        return {"val_loss": epoch_loss, "val_accuracy": epoch_accuracy}

    def epoch_end(self, epoch, result):
        print(f"Epoch [{epoch+1}], last_lr: {result['lrs'][-1]:.5f}, train_loss: {result['train_loss']:.4f}, val_loss: {result['val_loss']:.4f}, val_acc: {result['val_accuracy']:.4f}")

def ConvBlock(in_channels, out_channels, pool=False):
    layers = [
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    ]
    if pool:
        layers.append(nn.MaxPool2d(4)) # This is aggressive, common in some ResNet9 variants
        # Consider nn.MaxPool2d(2) if performance is poor, and adjust classifier accordingly
    return nn.Sequential(*layers)

class ResNet9(ImageClassificationBase):
    def __init__(self, in_channels, num_diseases):
        super().__init__()
        # Initial 256x256
        self.conv1 = ConvBlock(in_channels, 64)                     # out: 64 x 256 x 256
        self.conv2 = ConvBlock(64, 128, pool=True)                 # out: 128 x 64 x 64 (256/4=64)
        self.res1 = nn.Sequential(ConvBlock(128, 128), ConvBlock(128, 128)) # out: 128 x 64 x 64

        self.conv3 = ConvBlock(128, 256, pool=True)                # out: 256 x 16 x 16 (64/4=16)
        self.conv4 = ConvBlock(256, 512, pool=True)                # out: 512 x 4 x 4 (16/4=4)
        self.res2 = nn.Sequential(ConvBlock(512, 512), ConvBlock(512, 512)) # out: 512 x 4 x 4

        self.classifier = nn.Sequential(
            nn.MaxPool2d(4),                                       # out: 512 x 1 x 1 (4/4=1)
            # Alternative: nn.AdaptiveAvgPool2d((1,1)) # More robust to input size changes
            nn.Flatten(),
            nn.Linear(512, num_diseases)
        )

    def forward(self, xb):
        out = self.conv1(xb)
        out = self.conv2(out)
        out = self.res1(out) + out # Residual connection
        out = self.conv3(out)
        out = self.conv4(out)
        out = self.res2(out) + out # Residual connection
        out = self.classifier(out)
        return out

# --- TRAINING FUNCTIONS ---
def accuracy(outputs, labels):
    """Calculate accuracy"""
    _, preds = torch.max(outputs, dim=1)
    return torch.tensor(torch.sum(preds == labels).item() / len(preds))

@torch.no_grad()
def evaluate(model, val_loader):
    """Evaluate the model on validation data"""
    model.eval()
    outputs = [model.validation_step(batch) for batch in val_loader]
    return model.validation_epoch_end(outputs)

def get_lr(optimizer):
    """Get the current learning rate"""
    for param_group in optimizer.param_groups:
        return param_group['lr']

def fit_one_cycle(epochs, max_lr, model, train_loader, val_loader,
                  weight_decay=0, grad_clip=None, opt_func=torch.optim.SGD,
                  early_stopping_patience=5): # Added early stopping
    """Train the model using the 1Cycle policy with early stopping"""
    torch.cuda.empty_cache()
    history = []

    optimizer = opt_func(model.parameters(), max_lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr, epochs=epochs, steps_per_epoch=len(train_loader)
    )

    best_val_loss = float('inf')
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []
        lrs = []

        print(f"\nEpoch {epoch+1}/{epochs}")
        # Progress bar for training
        train_bar = tqdm(train_loader, desc=f"Train Epoch {epoch+1}/{epochs}")
        running_loss = 0.0

        for i, batch in enumerate(train_bar):
            loss = model.training_step(batch)
            train_losses.append(loss.detach()) # Detach to free graph
            running_loss += loss.item()

            loss.backward()
            if grad_clip:
                nn.utils.clip_grad_value_(model.parameters(), grad_clip)
            optimizer.step()
            optimizer.zero_grad()

            lrs.append(get_lr(optimizer))
            sched.step()
            train_bar.set_postfix(loss=running_loss/(i+1), lr=lrs[-1])

        # Validation phase
        model.eval() # Ensure model is in evaluation mode
        val_outputs = [model.validation_step(batch) for batch in tqdm(val_loader, desc=f"Valid Epoch {epoch+1}/{epochs}")]
        result = model.validation_epoch_end(val_outputs)
        result['train_loss'] = torch.stack(train_losses).mean().item()
        result['lrs'] = lrs
        model.epoch_end(epoch, result) # Pass epoch (0-indexed)
        history.append(result)

        # Early stopping
        current_val_loss = result['val_loss'].item()
        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            epochs_no_improve = 0
            # Save the best model
            torch.save(model.state_dict(), 'best_plant_disease_model.pth')
            print(f"Validation loss improved to {best_val_loss:.4f}. Saved best model.")
        else:
            epochs_no_improve += 1
            print(f"Validation loss did not improve for {epochs_no_improve} epoch(s). Best: {best_val_loss:.4f}")

        if epochs_no_improve >= early_stopping_patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            # Load the best model weights before returning
            model.load_state_dict(torch.load('best_plant_disease_model.pth'))
            print("Loaded best model weights.")
            break
    return history


# --- PREDICTION FUNCTIONS ---
def predict_image(img_tensor, model, classes_list, device): # Use img_tensor and classes_list
    """Predict the class of a single image tensor"""
    model.eval() # Ensure model is in eval mode
    xb = to_device(img_tensor.unsqueeze(0), device)
    with torch.no_grad():
        yb = model(xb)
    _, preds = torch.max(yb, dim=1)
    return classes_list[preds[0].item()]

def predict_multiple(dataset_to_predict, model, classes_list, device, num_samples=5): # Use dataset and classes_list
    """Predict multiple images and show results"""
    if len(dataset_to_predict) == 0:
        print("Dataset for prediction is empty.")
        return

    actual_num_samples = min(num_samples, len(dataset_to_predict))
    if actual_num_samples == 0:
        print("No samples to predict.")
        return

    test_indices = torch.randperm(len(dataset_to_predict))[:actual_num_samples]

    plt.figure(figsize=(3 * actual_num_samples, 5)) # Adjusted figsize
    for i, idx in enumerate(test_indices):
        img_tensor, label_idx = dataset_to_predict[idx] # img is already a tensor

        # For display, denormalize
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img_display = img_tensor * std + mean
        img_display = torch.clamp(img_display, 0, 1)


        plt.subplot(1, actual_num_samples, i + 1)
        plt.imshow(img_display.permute(1, 2, 0))
        plt.axis('off')

        predicted_class_name = predict_image(img_tensor, model, classes_list, device)
        actual_class_name = classes_list[label_idx]

        color = 'green' if predicted_class_name == actual_class_name else 'red'
        title_text = f"Pred: {predicted_class_name.split('___')[-1]}\nActual: {actual_class_name.split('___')[-1]}"
        plt.title(title_text, color=color, fontsize=9) # Adjusted fontsize

    plt.tight_layout()
    plt.show()

# --- CSV CREATION ---
def create_dataset_csv(base_dir_for_csv, output_data_dir): # Renamed for clarity
    """Create a CSV file with plant and disease information from a base directory (e.g., train_dir or data_dir)"""
    image_paths = []
    plant_types = []
    disease_names = []
    class_names_list = [] # Renamed to avoid conflict

    # Walk through the directory structure (assuming it's train_dir or similar)
    if not os.path.isdir(base_dir_for_csv):
        print(f"Directory not found for CSV creation: {base_dir_for_csv}")
        return

    for class_folder in sorted(os.listdir(base_dir_for_csv)): # Sort for consistency
        class_path = os.path.join(base_dir_for_csv, class_folder)
        if os.path.isdir(class_path):
            parts = class_folder.split('___')
            plant = parts[0].replace('_', ' ')
            disease = parts[1].replace('_', ' ') if len(parts) > 1 else "Healthy"

            for img_file in sorted(os.listdir(class_path)): # Sort for consistency
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    full_path = os.path.join(class_path, img_file)
                    image_paths.append(full_path)
                    plant_types.append(plant)
                    disease_names.append(disease)
                    class_names_list.append(class_folder)

    if not image_paths:
        print(f"No images found in {base_dir_for_csv} to create CSV.")
        return

    df = pd.DataFrame({
        'image_path': image_paths,
        'plant_type': plant_types,
        'disease_name': disease_names,
        'class_name': class_names_list,
        'filename': [os.path.basename(p) for p in image_paths]
    })

    csv_path = os.path.join(output_data_dir, "plant_disease_dataset_info.csv") # Changed name slightly
    df.to_csv(csv_path, index=False)
    print(f"Dataset info CSV file created at: {csv_path}")

    unique_diseases = sorted(list(df['class_name'].unique())) # Get unique class names for labels
    print("\nClass Labels (from CSV data):")
    for i, cls_name in enumerate(unique_diseases):
        print(f"Label {i}: {cls_name}")


# --- MAIN EXECUTION ---
def main():
    print("=" * 50)
    print("PLANT DISEASE CLASSIFICATION")
    print("=" * 50)

    # Setup and download data
    data_dir, train_dir, valid_dir = setup_kaggle()
    if data_dir is None:
        print("Exiting due to Kaggle setup issues.")
        return

    print("\n" + "=" * 50)
    print("DATA EXPLORATION")
    print("=" * 50)
    diseases, nums, plants, img_per_class = explore_data(train_dir)

    print("\n" + "=" * 50)
    print("DATA PREPARATION")
    print("=" * 50)

    IMG_SIZE = 256 # Define image size
    # Data transformation with MORE augmentation
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)), # More robust
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15), # Slightly more rotation
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05), # Added saturation and hue
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), shear=5), # Small affine transforms
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # ImageNet stats
    ])

    valid_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)), # Ensure validation images are consistently sized
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    print("\nLoading training dataset...")
    train_dataset = ImageFolder(train_dir, transform=train_transform)
    print("Loading validation dataset...")
    valid_dataset = ImageFolder(valid_dir, transform=valid_transform)

    print(f"\nTraining dataset size: {len(train_dataset):,} images")
    print(f"Validation dataset size: {len(valid_dataset):,} images")
    # Ensure classes are consistent, ImageFolder sorts them by default
    if train_dataset.classes != valid_dataset.classes:
        print("Warning: Training and validation dataset classes do not match or are not in the same order!")
    print(f"Number of classes: {len(train_dataset.classes)}")
    # print("Classes:", train_dataset.classes) # Useful for debugging

    batch_size = 32 # Can try 64 if GPU memory allows
    print(f"\nCreating data loaders with batch size {batch_size}...")
    train_dl = DataLoader(train_dataset, batch_size, shuffle=True, num_workers=2, pin_memory=True)
    valid_dl = DataLoader(valid_dataset, batch_size, num_workers=2, pin_memory=True)

    print("\n" + "=" * 50)
    print("DATA VISUALIZATION")
    print("=" * 50)
    show_batch_fn = visualize_data(nums, diseases, train_dataset) # Pass train_dataset for samples
    show_batch_fn(train_dl) # Show a batch from train_dl

    print("\n" + "=" * 50)
    print("MODEL SETUP")
    print("=" * 50)
    device = get_device()
    print(f"\nUsing device: {device}")

    print("\nMoving data loaders to device...")
    train_dl = DeviceDataLoader(train_dl, device)
    valid_dl = DeviceDataLoader(valid_dl, device)

    print("\nInitializing ResNet9 model...")
    model = to_device(ResNet9(3, len(train_dataset.classes)), device)

    print("\nModel Architecture:")
    try:
        summary(model, (3, IMG_SIZE, IMG_SIZE))
    except Exception as e:
        print(f"Could not print model summary: {e}")


    print("\n" + "=" * 50)
    print("MODEL TRAINING")
    print("=" * 50)

    epochs = 8 # INCREASED EPOCHS
    max_lr = 0.01 # Keep for now, can adjust later
    grad_clip = 0.1
    weight_decay = 1e-4
    opt_func = torch.optim.Adam
    early_stopping_patience = 5 # Stop if val_loss doesn't improve for 5 epochs

    print(f"\nTraining for up to {epochs} epochs with:")
    print(f"- Max learning rate: {max_lr}")
    print(f"- Weight decay: {weight_decay}")
    print(f"- Gradient clipping: {grad_clip}")
    print(f"- Optimizer: {opt_func.__name__}")
    print(f"- Early stopping patience: {early_stopping_patience} epochs")

    print("\nStarting training...\n")
    history = fit_one_cycle(
        epochs, max_lr, model, train_dl, valid_dl,
        weight_decay, grad_clip, opt_func, early_stopping_patience
    )

    print("\n" + "=" * 50)
    print("TRAINING RESULTS")
    print("=" * 50)

    if not history:
        print("Training did not produce any history. Exiting.")
        return

    plt.figure(figsize=(12, 10)) # Adjusted figsize

    plt.subplot(3, 1, 1) # Changed to 3 plots for LRs
    train_losses = [x['train_loss'] for x in history]
    val_losses = [x['val_loss'].item() for x in history] # .item() for scalar tensor
    plt.plot(train_losses, '-bx', label='Training loss')
    plt.plot(val_losses, '-rx', label='Validation loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss', fontsize=14)
    plt.legend(fontsize=12)
    plt.grid(True)

    plt.subplot(3, 1, 2)
    accuracies = [x['val_accuracy'].item() for x in history] # .item() for scalar tensor
    plt.plot(accuracies, '-gx', label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Validation Accuracy', fontsize=14)
    plt.legend(fontsize=12)
    plt.grid(True)

    plt.subplot(3, 1, 3)
    lrs = np.concatenate([x.get('lrs', []) for x in history]) # Handle if lrs key missing
    plt.plot(lrs)
    plt.xlabel('Batch no.')
    plt.ylabel('Learning rate')
    plt.title('Learning Rate Schedule', fontsize=14)
    plt.grid(True)

    plt.tight_layout()
    plt.show()

    if accuracies:
        print("\nFinal validation accuracy (from best model): {:.2f}%".format(max(accuracies) * 100))
        # The model loaded at the end of fit_one_cycle is the best one if early stopping occurred.
    else:
        print("No accuracy data to report.")


    print("\n" + "=" * 50)
    print("MODEL EVALUATION (on validation set samples)")
    print("=" * 50)
    print("\nGenerating predictions on validation data samples...")
    # Use valid_dataset which has original (non-augmented beyond resize/normalize) images
    # and train_dataset.classes for the class names
    predict_multiple(valid_dataset, model, train_dataset.classes, device, num_samples=10)

    # Save the final (potentially best) model state
    final_model_path = 'final_plant_disease_model.pth'
    print(f"\nSaving final model to '{final_model_path}' (this might be the best model if early stopping occurred)...")
    torch.save(model.state_dict(), final_model_path)
    print("Model saved successfully!")

    print("\nCreating dataset CSV file from training data directory...")
    create_dataset_csv(train_dir, data_dir if data_dir else ".") # Use data_dir or current if not set

    print("\n" + "=" * 50)
    print("PROCESS COMPLETED SUCCESSFULLY!")
    print("=" * 50)

if __name__ == '__main__':
    # This ensures tqdm works correctly if run as a script outside a notebook
    from tqdm import tqdm as script_tqdm
    # Monkey patch tqdm.notebook.tqdm to use the standard tqdm if not in a notebook environment
    try:
        from IPython import get_ipython
        if get_ipython() is None or 'IPKernelApp' not in get_ipython().config:
            class TqdmNotebookWrapper:
                def __init__(self, *args, **kwargs):
                    self.instance = script_tqdm(*args, **kwargs)
                def __getattr__(self, name):
                    return getattr(self.instance, name)
                def __iter__(self):
                    return iter(self.instance)
            tqdm_notebook_module = __import__('tqdm.notebook', fromlist=['tqdm'])
            tqdm_notebook_module.tqdm = TqdmNotebookWrapper
    except (ImportError, AttributeError):
        pass # Not in an IPython environment or tqdm.notebook not used directly

    main()

ModuleNotFoundError: No module named 'tqdm'